# 03 · Charts
Each chart is saved as `charts/<name>.png` (static) and `charts/<name>.html` (interactive).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()
import plotly.graph_objects as go
from viz import add_caption, save, SERIES, FUEL_COLORS, INK, INK_2

## 1. Load vs temperature

In [2]:
m = q("SELECT avg_temp_c, demand_mwh FROM model_hourly WHERE demand_mwh IS NOT NULL")
binned = m.assign(t=m.avg_temp_c.round()).groupby("t").demand_mwh.mean().reset_index()
low = binned.loc[binned.demand_mwh.idxmin()]
fig = go.Figure()
fig.add_scattergl(x=m.avg_temp_c, y=m.demand_mwh / 1000, mode="markers", name="Hourly load",
                  marker=dict(size=3, color=SERIES[0], opacity=0.12),
                  hovertemplate="%{x:.1f} °C<br>%{y:.1f} GW<extra></extra>")
fig.add_scatter(x=binned.t, y=binned.demand_mwh / 1000, mode="lines", name="Average per 1 °C",
                line=dict(color=SERIES[1], width=3), hovertemplate="%{x:.0f} °C<br>avg %{y:.1f} GW<extra></extra>")
fig.add_annotation(x=low.t, y=low.demand_mwh / 1000, text=f"Lowest average load near {low.t:.0f} °C",
                   showarrow=True, arrowcolor=INK_2, ay=60, font=dict(color=INK))
fig.add_annotation(x=-11, y=47, text="Below −9 °C: Uri hours,<br>load shed, not true demand", showarrow=False,
                   xanchor="center", yanchor="top", yshift=-14, font=dict(color=INK_2, size=12))
fig.update_layout(title="ERCOT hourly load vs temperature, 2019–2025",
                  xaxis_title="Load-weighted temperature, 8 Texas cities (°C)", yaxis_title="Hourly load (GW)")
add_caption(fig, "Load is U-shaped in temperature: it bottoms out in mild weather and climbs fastest in summer heat.")
save(fig, "load_vs_temperature")

## 2. Net load and the duck curve
Average spring-day (March–May) net load profile, comparing three years.

In [3]:
n = q("""SELECT year, local_hour, AVG(net_load_mwh) / 1000 AS net_gw
         FROM net_load_hourly WHERE month(local_ts) IN (3, 4, 5) AND year IN (2019, 2022, 2025)
         GROUP BY year, local_hour ORDER BY year, local_hour""")
fig = go.Figure()
for i, (yr, g) in enumerate(n.groupby("year")):
    fig.add_scatter(x=g.local_hour, y=g.net_gw, mode="lines", name=str(yr), line=dict(color=SERIES[i], width=2.5),
                    hovertemplate=f"{yr} · %{{x}}:00<br>%{{y:.1f}} GW<extra></extra>")
    fig.add_annotation(x=23, y=g.net_gw.iloc[-1], text=str(yr), showarrow=False, xanchor="left", xshift=6,
                       yshift={2019: 9, 2022: -9}.get(yr, 0), font=dict(color=INK))
r25 = n[n.year == 2025].set_index("local_hour").net_gw
fig.add_annotation(x=r25.idxmin(), y=r25.min(), text=f"2025 midday trough {r25.min():.0f} GW", showarrow=True, ay=40, arrowcolor=INK_2, font=dict(color=INK))
fig.update_layout(title="The duck curve: average spring-day net load (demand minus wind and solar)",
                  xaxis_title="Hour ending (Central time)", yaxis_title="Net load (GW)", xaxis=dict(dtick=3, range=[0, 24.5]))
ramp = q("SELECT * FROM evening_ramp_yearly")
add_caption(fig, f"Midday net load has sunk as solar grew; the average evening 3-hour ramp rose from {ramp.avg_daily_max_ramp_mwh.iloc[0]/1000:.1f} GW (2019) to {ramp.avg_daily_max_ramp_mwh.iloc[-1]/1000:.1f} GW (2025).")
save(fig, "duck_curve")

## 3. Winter Storm Uri

In [4]:
u = q("""SELECT local_ts, demand_mwh, gas_mwh, coal_mwh, nuclear_mwh, wind_mwh, solar_mwh,
                COALESCE(hydro_mwh,0) + COALESCE(other_mwh,0) AS other_mwh, avg_temp_c, ercot_forecast_mwh
         FROM model_hourly WHERE local_date BETWEEN DATE '2021-02-11' AND DATE '2021-02-21' ORDER BY period""")
fig = go.Figure()
stack = [("Nuclear", "nuclear_mwh"), ("Coal", "coal_mwh"), ("Gas", "gas_mwh"), ("Wind", "wind_mwh"), ("Solar", "solar_mwh"), ("Other", "other_mwh")]
for name, col in stack:
    fig.add_scatter(x=u.local_ts, y=u[col] / 1000, name=name, stackgroup="gen", mode="lines",
                    line=dict(width=0.5, color="#fcfcfb"), fillcolor=FUEL_COLORS[name],
                    hovertemplate=f"{name}: %{{y:.1f}} GW<extra></extra>")
fig.add_scatter(x=u.local_ts, y=u.ercot_forecast_mwh / 1000, name="ERCOT day-ahead forecast", mode="lines",
                line=dict(color=INK, width=2, dash="dot"), hovertemplate="ERCOT forecast: %{y:.1f} GW<extra></extra>")
fig.add_scatter(x=u.local_ts, y=u.demand_mwh / 1000, name="Served demand", mode="lines",
                line=dict(color=INK, width=2.5), hovertemplate="Demand: %{y:.1f} GW<extra></extra>")
pk = u.loc[u.demand_mwh.idxmax()]
fig.add_annotation(x=pk.local_ts, y=pk.demand_mwh / 1000, text=f"Peak {pk.demand_mwh/1000:.1f} GW", showarrow=True, ay=-30, ax=-70, arrowcolor=INK_2, font=dict(color=INK))
fig.add_vrect(x0="2021-02-15 01:00", x1="2021-02-18 23:00", fillcolor="#8a8984", opacity=0.08, line_width=0,
              annotation_text="Rolling outages: served demand capped by supply", annotation_position="top left",
              annotation_font=dict(color=INK_2))
mid = u[u.local_ts == "2021-02-12 12:00"].iloc[0]
for name, col in [("Nuclear", "nuclear_mwh"), ("Coal", "coal_mwh"), ("Gas", "gas_mwh"), ("Wind", "wind_mwh")]:
    y = (mid[[c for _, c in stack][:[c for _, c in stack].index(col) + 1]].sum() - mid[col] / 2) / 1000
    fig.add_annotation(x=mid.local_ts, y=y, text=name, showarrow=False, font=dict(color="white", size=13))
fig.update_layout(title="Winter Storm Uri, Feb 11–21 2021: ERCOT generation by fuel vs demand",
                  xaxis_title="Date (Central time)", yaxis_title="Hourly energy (GW)", yaxis=dict(range=[0, 85]))
out = u[(u.local_ts >= "2021-02-15") & (u.local_ts < "2021-02-19")]
gap = (out.ercot_forecast_mwh - out.demand_mwh).clip(lower=0)
add_caption(fig, f"Generation fell nearly 40% after the Feb 14 peak. ERCOT's own forecast ran up to {gap.max()/1000:.0f} GW above served demand, an upper bound on load shed.")
save(fig, "uri_week")